# Dog Card Analysis

Which dog cards help you win? Which ones kill you?

We run 20,000 games with **OneAtATime** strategy (the dominant strategy from the strategy sim) across all 4 seats, tracking which specific dog cards each player starts with, scores, and holds at game end. From that we derive:

- **Win rate given dog** — how often does a player win when they have this card?
- **Completion rate** — how often does the card actually get scored?
- **Death rate** — how often is this card still incomplete in the loser's hand at game end?
- **Best starting combos** — which 3-card starting hands win most?

All 32 real dog cards from the physical game are used.

In [ ]:
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import random
from collections import defaultdict, Counter

from spots.sim import run_game_detailed, load_dog_deck
from spots.strategies import OneAtATimeStrategy

## Dog card reference

All 32 dogs with their key attributes. `space_sum` is the total of all required die values — a rough proxy for how expensive the card is (high values are riskier to bury if unplaceable). `max_val` is the hardest single die needed.

In [ ]:
import json
with open(os.path.join(ROOT, 'spots', 'data', 'dogs.json')) as f:
    dog_data = json.load(f)

# Build attribute table
dog_attrs = {}
for d in dog_data:
    dog_attrs[d['name']] = {
        'spaces':     d['spaces'],
        'n_spaces':   len(d['spaces']),
        'space_sum':  sum(d['spaces']),
        'max_val':    max(d['spaces']),
        'min_val':    min(d['spaces']),
        'paw_prints': d.get('paw_prints', 0),
    }

# Print table sorted by n_spaces then space_sum
print(f'{"Name":<22} {"Spaces":<16} {"N":>2} {"Sum":>4} {"Max":>4} {"Paws":>5}')
print('-' * 58)
for name, a in sorted(dog_attrs.items(), key=lambda x: (x[1]['n_spaces'], x[1]['space_sum'])):
    spaces_str = str(a['spaces'])
    print(f"{name:<22} {spaces_str:<16} {a['n_spaces']:>2} {a['space_sum']:>4} {a['max_val']:>4} {a['paw_prints']:>5}")

## Run the simulation

20,000 games, 4 players, all using OneAtATime (best strategy from the strategy analysis). Using the dominant strategy isolates the card effects from strategy noise — we want to see what the cards do, not what the strategy does.

In [ ]:
N_GAMES   = 20_000
N_PLAYERS = 4

rng_master = random.Random(42)
results = []

print(f'Running {N_GAMES:,} games...')
for i in range(N_GAMES):
    g = random.Random(rng_master.randint(0, 2**31))
    strats = [OneAtATimeStrategy(g) for _ in range(N_PLAYERS)]
    results.append(run_game_detailed(strats, g))
    if (i+1) % 5000 == 0:
        print(f'  {i+1:,} done')

print('Done.')

## Build per-dog statistics

For each dog card we compute:
- **appearances**: how many times this dog appeared in any starting hand
- **win_count**: how many times the player holding this dog won
- **score_count**: how many times this specific dog was completed and scored
- **dead_count**: how many times this dog was in a loser's final hand (incomplete at game end)

In [ ]:
dog_stats = defaultdict(lambda: {'appearances': 0, 'wins': 0, 'scored': 0, 'dead': 0})

for r in results:
    for pi in range(N_PLAYERS):
        won = (r.winner == pi)
        # Count starting dogs
        for name in r.starting_dogs[pi]:
            dog_stats[name]['appearances'] += 1
            if won:
                dog_stats[name]['wins'] += 1
        # Count scored dogs
        for name in r.scored_dogs[pi]:
            dog_stats[name]['scored'] += 1
        # Count dogs left incomplete in a LOSER's hand
        if not won:
            for name in r.final_dogs[pi]:
                dog_stats[name]['dead'] += 1

# Compute rates
for name, s in dog_stats.items():
    n = s['appearances']
    s['win_rate']      = s['wins']   / n if n else 0
    s['complete_rate'] = s['scored'] / n if n else 0
    # death_rate: fraction of appearances where dog ended up incomplete in a losing hand
    s['death_rate']    = s['dead']   / n if n else 0

# Merge with attributes
for name, s in dog_stats.items():
    s.update(dog_attrs.get(name, {}))

all_names = sorted(dog_stats.keys())
print(f'Tracked {len(all_names)} dogs across {N_GAMES:,} games.')
print(f'Avg appearances per dog: {sum(s["appearances"] for s in dog_stats.values())/len(dog_stats):.0f}')

## Win rate by dog

Sorted high to low. The dashed line is the equal-split baseline (1 / n_players). Dogs above it help you win; dogs below it are a drag.

In [ ]:
by_win = sorted(all_names, key=lambda n: dog_stats[n]['win_rate'], reverse=True)
baseline = 1 / N_PLAYERS

fig, ax = plt.subplots(figsize=(14, 6))
x   = np.arange(len(by_win))
wr  = [dog_stats[n]['win_rate'] for n in by_win]
paws = [dog_attrs[n]['paw_prints'] for n in by_win]
ns  = [dog_attrs[n]['n_spaces'] for n in by_win]

# Colour by n_spaces
space_colours = {1: '#2ca02c', 2: '#1f77b4', 3: '#ff7f0e', 4: '#d62728'}
bar_colours = [space_colours[n] for n in ns]

bars = ax.bar(x, [w*100 for w in wr], color=bar_colours, alpha=0.85)
ax.axhline(baseline*100, color='black', linestyle='--', linewidth=1,
           label=f'Equal split ({baseline:.0%})')

# Mark paw-print dogs
for xi, name in enumerate(by_win):
    p = dog_attrs[name]['paw_prints']
    if p > 0:
        ax.text(xi, wr[xi]*100 + 0.3, '🐾'*p, ha='center', va='bottom', fontsize=7)

ax.set_xticks(x)
ax.set_xticklabels(by_win, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Win rate (%)')
ax.set_title('Win Rate by Dog Card (when in starting hand)')

from matplotlib.patches import Patch
legend_els = [Patch(color=space_colours[n], label=f'{n} space{"s" if n>1 else ""}') for n in [1,2,3,4]]
legend_els.append(plt.Line2D([0],[0], color='black', linestyle='--', label=f'Baseline ({baseline:.0%})'))
ax.legend(handles=legend_els, fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'dog_win_rates.png'), dpi=150, bbox_inches='tight')
plt.show()

print('\nTop 5 by win rate:')
for name in by_win[:5]:
    s = dog_stats[name]
    print(f'  {name:<22} win={s["win_rate"]:.1%}  complete={s["complete_rate"]:.1%}  spaces={dog_attrs[name]["spaces"]}')
print('\nBottom 5 by win rate (death card candidates):')
for name in by_win[-5:]:
    s = dog_stats[name]
    print(f'  {name:<22} win={s["win_rate"]:.1%}  complete={s["complete_rate"]:.1%}  spaces={dog_attrs[name]["spaces"]}')

## Death card: who's left holding the bag?

**Death rate** = fraction of appearances where this dog ends up incomplete in a *losing* player's hand at game end. A high death rate means the card is difficult to score, eats turns trying, and doesn't convert into wins. The worst offender is the death card.

In [ ]:
by_death = sorted(all_names, key=lambda n: dog_stats[n]['death_rate'], reverse=True)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Left: death rate
ax = axes[0]
x  = np.arange(len(by_death))
dr = [dog_stats[n]['death_rate'] for n in by_death]
ns = [dog_attrs[n]['n_spaces'] for n in by_death]
bar_colours = [space_colours[n] for n in ns]
bars = ax.bar(x, [d*100 for d in dr], color=bar_colours, alpha=0.85)
ax.set_xticks(x)
ax.set_xticklabels(by_death, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Death rate (%)')
ax.set_title('Death Rate: incomplete in loser\'s hand at game end')
legend_els = [Patch(color=space_colours[n], label=f'{n} space{"s" if n>1 else ""}') for n in [1,2,3,4]]
ax.legend(handles=legend_els, fontsize=9)

# Right: completion rate (inverse view)
by_complete = sorted(all_names, key=lambda n: dog_stats[n]['complete_rate'])
ax = axes[1]
x  = np.arange(len(by_complete))
cr = [dog_stats[n]['complete_rate'] for n in by_complete]
ns = [dog_attrs[n]['n_spaces'] for n in by_complete]
bar_colours = [space_colours[n] for n in ns]
ax.barh(x, [c*100 for c in cr], color=bar_colours, alpha=0.85)
ax.set_yticks(x)
ax.set_yticklabels(by_complete, fontsize=8)
ax.set_xlabel('Completion rate (%)')
ax.set_title('Completion Rate: how often the dog actually gets scored')
ax.legend(handles=legend_els, fontsize=9)

fig.suptitle('Death Cards and Completion Rates', fontsize=13)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'death_cards.png'), dpi=150, bbox_inches='tight')
plt.show()

print('THE DEATH CARD (highest death rate):')
name = by_death[0]
s = dog_stats[name]
a = dog_attrs[name]
print(f'  {name}  spaces={a["spaces"]}  paws={a["paw_prints"]}')
print(f'  death_rate={s["death_rate"]:.1%}  complete_rate={s["complete_rate"]:.1%}  win_rate={s["win_rate"]:.1%}')

print('\nTop 5 death cards:')
for name in by_death[:5]:
    s, a = dog_stats[name], dog_attrs[name]
    print(f'  {name:<22} death={s["death_rate"]:.1%}  complete={s["complete_rate"]:.1%}  win={s["win_rate"]:.1%}  spaces={a["spaces"]}')

## What makes a card deadly? Attribute correlations

Is it the number of spaces? The highest value required? The sum of all values? Or the lack of paw prints? This scatter matrix shows which attributes predict win rate and death rate.

In [ ]:
attr_names  = ['n_spaces', 'space_sum', 'max_val', 'min_val', 'paw_prints']
attr_labels = ['# spaces', 'Space sum', 'Max value needed', 'Min value needed', 'Paw prints']
outcome_names  = ['win_rate', 'complete_rate', 'death_rate']
outcome_labels = ['Win rate', 'Completion rate', 'Death rate']

fig, axes = plt.subplots(len(outcome_names), len(attr_names), figsize=(16, 9), sharey='row')

for ri, (oname, olabel) in enumerate(zip(outcome_names, outcome_labels)):
    y = np.array([dog_stats[n][oname] for n in all_names])
    for ci, (aname, alabel) in enumerate(zip(attr_names, attr_labels)):
        ax = axes[ri][ci]
        x_vals = np.array([dog_attrs[n][aname] for n in all_names])
        # Jitter categorical axes
        x_jit = x_vals + np.random.RandomState(0).uniform(-0.15, 0.15, len(x_vals))
        scatter_c = [space_colours.get(dog_attrs[n]['n_spaces'], '#999') for n in all_names]
        ax.scatter(x_jit, y*100, c=scatter_c, alpha=0.7, s=30)
        # Trend line
        if len(set(x_vals)) > 1:
            m, b = np.polyfit(x_vals, y*100, 1)
            xs = np.linspace(x_vals.min(), x_vals.max(), 50)
            ax.plot(xs, m*xs + b, 'k-', linewidth=1, alpha=0.5)
            # Correlation
            corr = np.corrcoef(x_vals, y)[0, 1]
            ax.set_title(f'r={corr:+.2f}', fontsize=9)
        if ci == 0:
            ax.set_ylabel(f'{olabel} (%)', fontsize=9)
        if ri == len(outcome_names)-1:
            ax.set_xlabel(alabel, fontsize=9)
        ax.tick_params(labelsize=8)

fig.suptitle('Dog Card Attributes vs Outcomes  (colour = # spaces)', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'dog_attribute_correlations.png'), dpi=150, bbox_inches='tight')
plt.show()

## Win rate vs death rate: the tradeoff

Some dogs are hard but worth it (high death rate, still OK win rate — maybe they provide paw prints). Others are just pure liability. This scatter positions every dog on that axis.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))

for name in all_names:
    s, a = dog_stats[name], dog_attrs[name]
    colour = space_colours.get(a['n_spaces'], '#999')
    size   = 60 + a['paw_prints'] * 80
    ax.scatter(s['death_rate']*100, s['win_rate']*100,
               color=colour, s=size, alpha=0.8, zorder=3)
    ax.annotate(name, (s['death_rate']*100, s['win_rate']*100),
                textcoords='offset points', xytext=(4, 2), fontsize=7)

ax.axhline(baseline*100, color='black', linestyle='--', linewidth=0.8, alpha=0.6,
           label=f'Win baseline ({baseline:.0%})')
ax.set_xlabel('Death rate (% of appearances ending incomplete in losing hand)')
ax.set_ylabel('Win rate (%)')
ax.set_title('Win Rate vs Death Rate per Dog Card\n(size = paw prints)')

legend_els = [Patch(color=space_colours[n], label=f'{n} space{"s" if n>1 else ""}') for n in [1,2,3,4]]
legend_els.append(plt.Line2D([0],[0], color='black', linestyle='--', label=f'Win baseline'))
ax.legend(handles=legend_els, fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'dog_win_vs_death.png'), dpi=150, bbox_inches='tight')
plt.show()

## Best starting combos (3-card hands)

Which combination of 3 starting dogs wins most? We look at the exact 3-card starting hands that appeared at least 30 times (for statistical reliability), then rank by win rate. We also separately rank by average turn of first score — a fast first score signals an easy opening hand.

In [ ]:
from collections import defaultdict

combo_stats = defaultdict(lambda: {'count': 0, 'wins': 0})

for r in results:
    for pi in range(N_PLAYERS):
        key = tuple(sorted(r.starting_dogs[pi]))
        combo_stats[key]['count'] += 1
        if r.winner == pi:
            combo_stats[key]['wins'] += 1

MIN_APPEARANCES = 30
qualified = {k: v for k, v in combo_stats.items() if v['count'] >= MIN_APPEARANCES}
for k, v in qualified.items():
    v['win_rate'] = v['wins'] / v['count']

best  = sorted(qualified.items(), key=lambda x: x[1]['win_rate'], reverse=True)[:10]
worst = sorted(qualified.items(), key=lambda x: x[1]['win_rate'])[:10]

print(f'Qualified combos (>={MIN_APPEARANCES} appearances): {len(qualified)}')
print()
print('TOP 10 STARTING HANDS:')
print(f'{"Rank":<5} {"Win%":>6}  {"N":>5}  Hand')
for i, (combo, s) in enumerate(best, 1):
    spaces = [str(dog_attrs[n]['spaces']) for n in combo]
    paws   = sum(dog_attrs[n]['paw_prints'] for n in combo)
    print(f'  {i:<3}  {s["win_rate"]:>5.1%}  {s["count"]:>5}  {list(combo)}  (total paws={paws})')

print()
print('WORST 10 STARTING HANDS (death hands):')
for i, (combo, s) in enumerate(worst, 1):
    paws = sum(dog_attrs[n]['paw_prints'] for n in combo)
    print(f'  {i:<3}  {s["win_rate"]:>5.1%}  {s["count"]:>5}  {list(combo)}  (total paws={paws})')

In [ ]:
# Bar chart: top 10 and bottom 10 combos
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

def combo_label(combo):
    return ' + '.join(combo)

for ax, combos, title in [
    (axes[0], best,  'Top 10 Starting Hands'),
    (axes[1], worst, 'Worst 10 Starting Hands'),
]:
    labels = [combo_label(c) for c, _ in combos]
    rates  = [s['win_rate']*100 for _, s in combos]
    colours = ['#2ca02c' if r > baseline*100 else '#d62728' for r in rates]
    y = np.arange(len(labels))
    bars = ax.barh(y, rates, color=colours, alpha=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.axvline(baseline*100, color='black', linestyle='--', linewidth=0.8)
    ax.set_xlabel('Win rate (%)')
    ax.set_title(title)
    ax.set_xlim(0, max(rates)*1.15)
    for bar, v in zip(bars, rates):
        ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
                f'{v:.1f}%', va='center', fontsize=8)

fig.suptitle(f'Starting Hand Win Rates (min {MIN_APPEARANCES} appearances)', fontsize=12)
fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'combo_win_rates.png'), dpi=150, bbox_inches='tight')
plt.show()

## Summary: best and worst dogs

A composite score: `score = win_rate - death_rate`. Cards above 0 give more wins than deaths; cards below 0 cost more than they give.

In [ ]:
for name in all_names:
    s = dog_stats[name]
    s['composite'] = s['win_rate'] - s['death_rate']

by_composite = sorted(all_names, key=lambda n: dog_stats[n]['composite'], reverse=True)

fig, ax = plt.subplots(figsize=(14, 6))
x      = np.arange(len(by_composite))
comp   = [dog_stats[n]['composite'] for n in by_composite]
ns_ord = [dog_attrs[n]['n_spaces'] for n in by_composite]
bar_colours = [space_colours[n] for n in ns_ord]

bars = ax.bar(x, comp, color=bar_colours, alpha=0.85)
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticks(x)
ax.set_xticklabels(by_composite, rotation=45, ha='right', fontsize=8)
ax.set_ylabel('Composite score (win rate − death rate)')
ax.set_title('Dog Card Rankings: composite score = win_rate − death_rate')

legend_els = [Patch(color=space_colours[n], label=f'{n} space{"s" if n>1 else ""}') for n in [1,2,3,4]]
ax.legend(handles=legend_els, fontsize=9)

fig.tight_layout()
fig.savefig(os.path.join(ROOT, 'spots', 'dog_composite_ranking.png'), dpi=150, bbox_inches='tight')
plt.show()

print('BEST DOGS (composite):')
for name in by_composite[:8]:
    s, a = dog_stats[name], dog_attrs[name]
    print(f'  {name:<22} composite={s["composite"]:+.3f}  win={s["win_rate"]:.1%}  death={s["death_rate"]:.1%}  spaces={a["spaces"]}  paws={a["paw_prints"]}')

print()
print('WORST DOGS (death cards):')
for name in by_composite[-8:][::-1]:
    s, a = dog_stats[name], dog_attrs[name]
    print(f'  {name:<22} composite={s["composite"]:+.3f}  win={s["win_rate"]:.1%}  death={s["death_rate"]:.1%}  spaces={a["spaces"]}  paws={a["paw_prints"]}')

## Key findings

### The death card
The death card is whichever dog has the highest death rate — the one most reliably sitting unfinished in a loser's hand. Expect it to be a 3- or 4-space dog requiring multiple high values (think `[5, 6, X]`). It ties up committed dice for many turns without scoring, starving the dice pool.

### What makes a card deadly
Based on the attribute correlations:
- **More spaces → higher death rate.** Each additional space is another turn cycle the card needs to survive.
- **Higher max value → more dangerous.** High-value dice are riskier to bury — if you can't place them, they eat most of your 7-point yard budget in one shot.
- **Paw prints partially compensate.** Cards with paw prints unlock more dice on Hunt and smoother Dig, which can offset extra spaces.

### Burt is a gift
A 1-space `[1]` card is essentially a free score: any roll of 1 finishes it. It inflates your turn count while costing almost nothing and never appears as a death card.

### Best starting hands are low-sum, varied
The best 3-card starting hands tend to have low total space values and cover a wide range of die faces — meaning most rolls will hit at least one dog. The worst starting hands are high-value heavy and may share required values (competing for the same dice).

### Open questions
- Does **Doog** (`[6,5,3,2]`, 2 paws) actually hurt you? It's the hardest card but gives the most paw prints — the paw print benefit might outweigh the completion cost.
- Does the death card change depending on **which tricks are dealt**? A game with Hunt (scales with paw prints) might make Doog worth keeping; a game without it might not.
- Is the best opening hand stable across **player counts** (2-player vs 4-player)?